# 05_4 — Comparação Unificada de Modelos

Este notebook **unifica** os três pipelines de classificação de eventos urbanos que antes viviam em arquivos separados:

| Origem | Algoritmo | XAI |
|--------|-----------|-----|
| `05_XAI.ipynb` | **LightGBM** (gradient boosting) | SHAP (TreeExplainer) |
| `05_2_FTTransformer.ipynb` | **FT-Transformer** (transformer tabular) | Importância por permutação |
| `05_3_TabTransformer.ipynb` | **TabTransformer** (transformer tabular) | Importância por permutação |

## Objetivo

Aplicar as duas dimensões experimentais que desenvolvemos (**janela temporal** e **distância ao aeroporto on/off**) a **todos os três algoritmos** e, ao final, comparar **todos contra todos** para identificar o melhor cenário.

O grid combinatório é:

$$3\ \text{algoritmos} \times 4\ \text{janelas} \times 2\ \text{opções de distância} = 24\ \text{experimentos}$$

## Como garantimos uma comparação JUSTA

Os três notebooks originais divergiam na engenharia de features (o LightGBM não usava sinal espacial, os transformers usavam sin/cos, etc.). Para que a comparação meça o **algoritmo** e não o pré-processamento, aqui todos consomem **exatamente a mesma Base Master** e o **mesmo conjunto de features** (adaptado apenas ao formato que cada modelo exige).

**Desvios conscientes em relação aos originais** (documentados para a tese):

1. **Feature engineering unificado:** todos recebem features espaciais (lat/lng/dist), cíclicas (sin/cos) e lags+diffs climáticos. Um filtro de variância zero remove colunas constantes (ex.: lat/long da estação meteorológica), preservando o sinal espacial derivado do H3 (que varia pela cidade).
2. **Pesos de classe:** usamos `compute_class_weight('balanced')` para os três. O reforço manual (classe 0 ×0.5, classes 1/2 ×1.5) do `05_XAI` foi **removido** — como os dados já estão 1:1:1, ele só introduziria viés não comparável.
3. **LightGBM sem Optuna por padrão:** o grid usa hiperparâmetros fixos e sólidos (`LGBM_PARAMS`) para manter o tempo controlável e a capacidade comparável. `LGBM_OPTUNA_TRIALS > 0` reativa a busca (lento: roda por cenário).

## Roteiro

1. Setup e instalações — 2. Parâmetros — 3. Carga de dados — 4. Balanceamento — 5. Feature engineering parametrizado — 6. Preparação por algoritmo — 7. Funções de treino/avaliação — 8. Loop combinatório (24 runs) — 9. Comparação geral — 10. XAI do melhor cenário — 11. Conclusões.

## 1. Setup e Instalações

Instala e importa **todas** as dependências dos três mundos num só lugar: `lightgbm`, `shap`, `optuna` (LightGBM + XAI) e `tab-transformer-pytorch` (FT/Tab), além de `gcsfs`/`duckdb` (leitura no GCS) e `h3` (decodificação espacial).

In [ ]:
!pip install -q gcsfs duckdb lightgbm shap optuna tab-transformer-pytorch h3

import numpy as np
import pandas as pd
import copy
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import lightgbm as lgb
from lightgbm import LGBMClassifier
import optuna
import shap
import h3

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from tab_transformer_pytorch import FTTransformer, TabTransformer

import matplotlib.pyplot as plt
import seaborn as sns

print('Imports OK | GPU disponivel:', torch.cuda.is_available())

## 2. Parâmetros do Experimento

Todos os controles do grid ficam aqui, num único ponto:

- **`JANELAS`** — tamanhos de janela de agregação temporal testados.
- **`DIST_OPTIONS`** — `True` inclui `dist_aeroporto_km`/`lat`/`lng` nas features; `False` as remove.
- **`ALGORITMOS`** — quais modelos rodar (remova um da lista para pular).
- **`EPOCHS`/`PATIENCE`/`BATCH`** — treino dos transformers.
- **`LGBM_OPTUNA_TRIALS`** — `0` usa `LGBM_PARAMS` fixos; `>0` roda Optuna por cenário (**lento**).

> Para um teste rápido, reduza `JANELAS` (ex.: `['4h']`) ou `ALGORITMOS`.

In [ ]:
RANDOM_STATE = 42

# ---- Grid combinatorio ----
JANELAS      = ['1h', '2h', '3h', '4h']
DIST_OPTIONS = [True, False]
ALGORITMOS   = ['LightGBM', 'FT-Transformer', 'TabTransformer']

# ---- Treino dos transformers ----
EPOCHS   = 150
PATIENCE = 15
BATCH    = 512

# ---- LightGBM ----
LGBM_OPTUNA_TRIALS = 0   # 0 = params fixos (rapido) | >0 = Optuna por cenario (lento)
LGBM_PARAMS = dict(
    objective='multiclass', num_class=3, n_estimators=800, learning_rate=0.05,
    num_leaves=63, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
)

# ---- Rotulos ----
target_map    = {0: '0 (Nao Evento)', 1: '1 (Evento Climatico)', 2: '2 (Atraso de Voo)'}
nomes_classes = ['Nao Evento (0)', 'Clima (1)', 'Atraso Voo (2)']

print(f'Grid: {len(ALGORITMOS)} algoritmos x {len(JANELAS)} janelas x {len(DIST_OPTIONS)} dist = '
      f'{len(ALGORITMOS)*len(JANELAS)*len(DIST_OPTIONS)} experimentos')

## 3. Carga dos Dados (GCS + DuckDB + CSVs)

Pipeline idêntico ao dos três notebooks originais (executa no **Google Colab**):

1. Autentica no GCS e monta o Drive.
2. Copia os CSVs de apoio (clima/UTCI, voos atrasados SBPA, células H3 do aeroporto).
3. Lê os parquets das simulações `V6_3`..`V6_7` via DuckDB, amostrando 30% e classificando cada evento em `DS_VOO` / `DS_CLIMA` / `DS_OUTROS`.

In [ ]:
from google.colab import auth, drive
import gcsfs, duckdb, shutil

auth.authenticate_user()
project_id  = 'doutorado-501917'
bucket_name = '2025_rides'
fs = gcsfs.GCSFileSystem(project=project_id)
try:
    duckdb.register_filesystem(fs)
except Exception:
    pass

drive.mount('/content/drive')
datasets_dir = '/content/drive/MyDrive/DOUTORADO/DATASETS/DATASETS_PRONTOS'
base_dir     = '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/arquivos_base'
shutil.copy(f'{base_dir}/dados_meteorologicos_utci_horario.csv', './')
shutil.copy(f'{base_dir}/DADOS_AEROPORTO/03_voos_atrasados_sbpa.csv', './')
shutil.copy(f'{datasets_dir}/Aeroporto_Salgado_Filho_h3_res12.csv', './')

df_clima = pd.read_csv('/content/dados_meteorologicos_utci_horario.csv')
df_voos  = pd.read_csv('/content/03_voos_atrasados_sbpa.csv', sep=';')
df_h3    = pd.read_csv('/content/Aeroporto_Salgado_Filho_h3_res12.csv')
print('clima', df_clima.shape, '| voos', df_voos.shape, '| h3', df_h3.shape)

In [ ]:
caminhos_base = [f'gs://{bucket_name}/outputs_simulation_V6_{i}/trips_log/_staging/**/*.parquet'
                 for i in range(3, 8)]

todos_arquivos = []
for caminho in caminhos_base:
    todos_arquivos.extend([f'gs://{f}' for f in fs.glob(caminho)])
print(f'Total de {len(todos_arquivos):,} arquivos Parquet.')

files_sql_array = ', '.join([f"'{f}'" for f in todos_arquivos])
inicio_ts, fim_ts = 1735699200, 1767235200

query = f'''
    SELECT request_ts, event_name, origin_h3,
        CASE WHEN UPPER(event_name) LIKE '%ATRASADO%'  THEN 'DS_VOO'
             WHEN UPPER(event_name) LIKE '%SEVERIDADE%' THEN 'DS_CLIMA'
             WHEN event_name IS NULL                    THEN 'NULO'
             ELSE 'DS_OUTROS' END AS dataset_type
    FROM read_parquet([{files_sql_array}], hive_partitioning = true)
    WHERE request_ts >= {inicio_ts} AND request_ts < {fim_ts}
    USING SAMPLE 30 PERCENT
'''

df_combined = duckdb.sql(query).df()
DS_VOO    = df_combined[df_combined['dataset_type'] == 'DS_VOO'].drop(columns=['dataset_type'])
DS_CLIMA  = df_combined[df_combined['dataset_type'] == 'DS_CLIMA'].drop(columns=['dataset_type'])
DS_OUTROS = df_combined[df_combined['dataset_type'] == 'DS_OUTROS'].drop(columns=['dataset_type'])
print(f'Lidas {len(df_combined):,} linhas | DS_VOO={len(DS_VOO):,} DS_CLIMA={len(DS_CLIMA):,} DS_OUTROS={len(DS_OUTROS):,}')

## 4. Balanceamento (undersampling 1:1:1)

Filtra 2025, encontra a menor classe e faz *undersampling* aleatório (`random_state=42`) das outras duas até esse tamanho, preservando a ordem temporal. Resultado: três classes com exatamente o mesmo número de amostras.

In [ ]:
start_date = pd.to_datetime('2025-01-01 00:00:00')
end_date   = pd.to_datetime('2025-12-31 23:59:59')

for _d in [DS_VOO, DS_CLIMA, DS_OUTROS]:
    _d['request_ts_dt'] = pd.to_datetime(_d['request_ts'], unit='s')

def _janela_2025(d):
    return d[(d['request_ts_dt'] >= start_date) & (d['request_ts_dt'] <= end_date)]

DS_VOO, DS_CLIMA, DS_OUTROS = _janela_2025(DS_VOO), _janela_2025(DS_CLIMA), _janela_2025(DS_OUTROS)
min_size = min(len(DS_VOO), len(DS_CLIMA), len(DS_OUTROS))
print(f'Menor classe: {min_size:,} linhas')

def _balancear(d):
    if len(d) > min_size:
        d = d.sample(n=min_size, random_state=42)
    return d.sort_values('request_ts_dt').reset_index(drop=True)

DS_VOO    = _balancear(DS_VOO)
DS_CLIMA  = _balancear(DS_CLIMA)
DS_OUTROS = _balancear(DS_OUTROS)
print(f'Balanceado 1:1:1 -> {len(DS_VOO):,} cada | total {len(DS_VOO)+len(DS_CLIMA)+len(DS_OUTROS):,}')

## 5. Feature Engineering Parametrizado

A função `construir_base_master(window_size)` reconstrói a Base Master para uma dada janela. Ela **sempre** calcula todas as features (inclusive as espaciais); a decisão de usar ou não a distância acontece depois, na seleção de features. Isso torna o toggle de distância barato e evita recomputar a base.

Features geradas:
- **Espaciais:** H3 → lat/lng (cache computado uma vez) + `dist_aeroporto_km` (haversine até o Salgado Filho).
- **Climáticas:** média de todas as variáveis meteorológicas/UTCI por janela + contagem de voos/empresas.
- **Cíclicas:** sin/cos de hora, mês e dia da semana.
- **Temporais defasadas:** lag (janela anterior) e diff (variação) das principais variáveis de clima.

In [ ]:
# ---- Helpers espaciais ----
AER_LAT, AER_LNG = -29.9939, -51.1711   # Aeroporto Salgado Filho (SBPA)

def h3_to_latlng(h):
    try:
        return h3.cell_to_latlng(h)   # h3 v4
    except AttributeError:
        return h3.h3_to_geo(h)        # h3 v3

def haversine_km(lat1, lng1, lat2, lng2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi   = np.radians(lat2 - lat1)
    dlmb   = np.radians(lng2 - lng1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Cache de coordenadas H3 (computado UMA vez para todos os cenarios)
coords_cache = {}
for hx in pd.concat([DS_VOO, DS_CLIMA, DS_OUTROS])['origin_h3'].dropna().unique():
    try:
        coords_cache[hx] = h3_to_latlng(hx)
    except Exception:
        coords_cache[hx] = (np.nan, np.nan)
print(f'Cache H3: {len(coords_cache):,} celulas unicas')

def construir_base_master(window_size):
    dfc = df_clima.copy()
    dfv = df_voos.copy()
    dfc['time'] = pd.to_datetime(dfc['time'])
    dfv['CHEGADA_REAL'] = pd.to_datetime(dfv['CHEGADA_REAL'], errors='coerce')
    dfc['time_window'] = dfc['time'].dt.floor(window_size)
    dfv['time_window'] = dfv['CHEGADA_REAL'].dt.floor(window_size)

    dsv, dsc, dso = DS_VOO.copy(), DS_CLIMA.copy(), DS_OUTROS.copy()
    for _d, t in [(dso, 0), (dsc, 1), (dsv, 2)]:
        _d['request_ts_dt'] = pd.to_datetime(_d['request_ts_dt'])
        _d['time_window']   = _d['request_ts_dt'].dt.floor(window_size)
        _d['target']        = t

    df_ev = pd.concat([
        dso[['time_window', 'origin_h3', 'target', 'request_ts_dt']],
        dsc[['time_window', 'origin_h3', 'target', 'request_ts_dt']],
        dsv[['time_window', 'origin_h3', 'target', 'request_ts_dt']],
    ], ignore_index=True)

    clima_agg = dfc.drop(columns=['time']).groupby('time_window').mean(numeric_only=True).reset_index()
    voos_agg = dfv.dropna(subset=['time_window']).groupby('time_window').agg(
        qtd_voos_previstos  = ('NUMERO_VOO', 'count'),
        qtd_empresas_aereas = ('ICAO_EMPRESA_AEREA', lambda x: x.nunique()),
    ).reset_index()

    dm = pd.merge(df_ev, clima_agg, on='time_window', how='left')
    dm = pd.merge(dm, voos_agg, on='time_window', how='left')
    dm['qtd_voos_previstos']  = dm['qtd_voos_previstos'].fillna(0)
    dm['qtd_empresas_aereas'] = dm['qtd_empresas_aereas'].fillna(0)

    # Espacial (sempre calculado; o toggle acontece na selecao de features)
    dm['lat'] = dm['origin_h3'].map(lambda h: coords_cache.get(h, (np.nan, np.nan))[0])
    dm['lng'] = dm['origin_h3'].map(lambda h: coords_cache.get(h, (np.nan, np.nan))[1])
    dm['dist_aeroporto_km'] = haversine_km(dm['lat'], dm['lng'], AER_LAT, AER_LNG)

    # Temporal + ciclicas
    dm['hora']       = dm['time_window'].dt.hour
    dm['mes']        = dm['time_window'].dt.month
    dm['dia_semana'] = dm['time_window'].dt.dayofweek
    for col, period in [('hora', 24), ('mes', 12), ('dia_semana', 7)]:
        dm[f'{col}_sin'] = np.sin(2*np.pi * dm[col] / period)
        dm[f'{col}_cos'] = np.cos(2*np.pi * dm[col] / period)

    # Lags e diffs climaticos (janela anterior)
    cols_lag = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']
    if 'surface_pressure' in clima_agg.columns:
        cols_lag.append('surface_pressure')
    ca = clima_agg.sort_values('time_window').copy()
    for c in cols_lag:
        if c in ca.columns:
            ca[f'{c}_lag'] = ca[c].shift(1)
    lag_cols = [f'{c}_lag' for c in cols_lag if c in ca.columns]
    dm = pd.merge(dm, ca[['time_window'] + lag_cols], on='time_window', how='left')
    for c in cols_lag:
        if c in dm.columns and f'{c}_lag' in dm.columns:
            dm[f'diff_{c}'] = dm[c] - dm[f'{c}_lag']

    return dm.sort_values('time_window').reset_index(drop=True)

print('construir_base_master() pronta.')

## 6. Seleção de Features e Preparação por Algoritmo

`selecionar_features` define o **mesmo** conjunto de features contínuas para os três modelos:
- parte das colunas numéricas, remove IDs/alvo/temporais-categóricas,
- **descarta colunas de variância zero** (ex.: lat/long constantes da estação),
- e remove `lat`/`lng`/`dist_aeroporto_km` quando `use_dist=False`.

`preparar_dados` faz **um único split temporal** (80% treino / 20% teste; dentro do treino, últimos 15% = validação) reutilizado por todos os algoritmos, garantindo que comparam sobre exatamente os mesmos períodos. Retorna:
- matrizes escalonadas para os **transformers** (contínuas normalizadas + categóricas hora/mês/dia como tokens),
- DataFrames não escalonados para o **LightGBM** (árvores não precisam de normalização; hora/mês/dia entram como numéricas).

In [ ]:
IDS_EXCL = ['time_window', 'request_ts_dt', 'event_name', 'origin_h3', 'target', 'request_ts', 'trimestre']
CAT_COLS = ['hora', 'mes', 'dia_semana']
SPATIAL  = ['lat', 'lng', 'dist_aeroporto_km']

def selecionar_features(df, use_dist):
    num = df.select_dtypes(include=[np.number]).columns.tolist()
    cont = [c for c in num if c not in IDS_EXCL + CAT_COLS and not c.startswith('lista_')]
    var = df[cont].var(numeric_only=True)
    cont = [c for c in cont if var.get(c, 0) > 0]          # remove constantes
    if not use_dist:
        cont = [c for c in cont if c not in SPATIAL]
    return cont

def preparar_dados(df_master, use_dist):
    cont = selecionar_features(df_master, use_dist)
    n = len(df_master)
    corte  = int(n * 0.80)          # treino+val | teste
    vcorte = int(corte * 0.85)      # treino | validacao (dentro do treino)

    # Matrizes para transformers
    Xc = df_master[cont].fillna(0).values.astype('float32')
    Xk = np.stack([
        df_master['hora'].values.astype('int64'),
        (df_master['mes'].values - 1).astype('int64'),   # 1-12 -> 0-11
        df_master['dia_semana'].values.astype('int64'),
    ], axis=1)
    y = df_master['target'].values.astype('int64')

    scaler = StandardScaler().fit(Xc[:corte])             # fit so no treino
    Xc_s = scaler.transform(Xc).astype('float32')

    # DataFrame para LightGBM (cont NAO escalonado + categoricas como numericas)
    feat_lgbm = cont + CAT_COLS
    Xl = df_master[feat_lgbm].fillna(0)

    return {
        'cont': cont,
        # transformers
        'Xc_tr':  Xc_s[:vcorte],       'Xk_tr':  Xk[:vcorte],       'y_tr':  y[:vcorte],
        'Xc_val': Xc_s[vcorte:corte],  'Xk_val': Xk[vcorte:corte],  'y_val': y[vcorte:corte],
        'Xc_te':  Xc_s[corte:],        'Xk_te':  Xk[corte:],        'y_te':  y[corte:],
        # lightgbm
        'lgbm': {
            'features': feat_lgbm,
            'X_tr':   Xl.iloc[:vcorte],      'y_tr':   y[:vcorte],
            'X_val':  Xl.iloc[vcorte:corte], 'y_val':  y[vcorte:corte],
            'X_test': Xl.iloc[corte:],       'y_test': y[corte:],
        },
    }

print('selecionar_features() e preparar_dados() prontas.')

## 7. Funções de Treino e Avaliação

Uma função por família de modelo, todas retornando o **mesmo dicionário de métricas** (`accuracy`, `macro_f1`, e F1 por classe) mais objetos internos (prefixados com `_`) usados depois na XAI.

- `treinar_transformer(build_fn, ...)` serve **FT-Transformer e TabTransformer** (mesmo loop; muda só a classe do modelo, construída por `build_ft`/`build_tab`). Ambos recebem `model(x_categ, x_cont)` — **categóricas primeiro**.
- `treinar_lightgbm(...)` treina o LightGBM (params fixos ou Optuna).
- `importancia_permutacao(...)` calcula importância por permutação para os transformers.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CATEGORIES = (24, 12, 7)   # cardinalidades: hora(0-23), mes(0-11), dia_semana(0-6)

def copy_state(m):
    return copy.deepcopy({k: v.detach().cpu().clone() for k, v in m.state_dict().items()})

def build_ft(num_cont, categories):
    return FTTransformer(categories=categories, num_continuous=num_cont,
                         dim=32, dim_out=3, depth=6, heads=8,
                         attn_dropout=0.1, ff_dropout=0.1)

def build_tab(num_cont, categories):
    return TabTransformer(categories=categories, num_continuous=num_cont,
                          dim=32, dim_out=3, depth=6, heads=8,
                          attn_dropout=0.1, ff_dropout=0.1,
                          mlp_hidden_mults=(4, 2), mlp_act=nn.ReLU())

def _metricas(y_te, y_pred):
    f1c = f1_score(y_te, y_pred, average=None, zero_division=0)
    return {
        'accuracy':      round(float((y_pred == y_te).mean()), 4),
        'macro_f1':      round(float(f1_score(y_te, y_pred, average='macro', zero_division=0)), 4),
        'f1_nao_evento': round(float(f1c[0]), 4),
        'f1_clima':      round(float(f1c[1]), 4),
        'f1_atraso_voo': round(float(f1c[2]), 4),
    }

def treinar_transformer(build_fn, titulo, dados, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, verbose=False):
    to_t = lambda a, d: torch.tensor(a, dtype=d, device=device)
    tXc_tr,  tXk_tr,  ty_tr  = to_t(dados['Xc_tr'], torch.float32),  to_t(dados['Xk_tr'], torch.long),  to_t(dados['y_tr'], torch.long)
    tXc_val, tXk_val, ty_val = to_t(dados['Xc_val'], torch.float32), to_t(dados['Xk_val'], torch.long), to_t(dados['y_val'], torch.long)
    tXc_te,  tXk_te          = to_t(dados['Xc_te'], torch.float32),  to_t(dados['Xk_te'], torch.long)

    model = build_fn(len(dados['cont']), CATEGORIES).to(device)
    pesos = compute_class_weight('balanced', classes=np.unique(dados['y_tr']), y=dados['y_tr'])
    crit  = nn.CrossEntropyLoss(weight=torch.tensor(pesos, dtype=torch.float32, device=device), label_smoothing=0.05)
    opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=6, min_lr=1e-5)

    dl = DataLoader(TensorDataset(tXk_tr, tXc_tr, ty_tr), batch_size=batch, shuffle=True)
    best_val, best_state, wait = float('inf'), None, 0
    hist = {'loss': [], 'val_loss': [], 'acc': [], 'val_acc': []}

    for ep in range(1, epochs + 1):
        model.train(); tl = tc = tn = 0
        for xk, xc, yb in dl:
            opt.zero_grad(); out = model(xk, xc); loss = crit(out, yb); loss.backward(); opt.step()
            tl += loss.item() * len(yb); tc += (out.argmax(1) == yb).sum().item(); tn += len(yb)
        model.eval()
        with torch.no_grad():
            vo = model(tXk_val, tXc_val); vl = crit(vo, ty_val).item()
            va = (vo.argmax(1) == ty_val).float().mean().item()
        sched.step(vl)
        hist['loss'].append(tl/tn); hist['val_loss'].append(vl); hist['acc'].append(tc/tn); hist['val_acc'].append(va)
        if vl < best_val - 1e-4:
            best_val, best_state, wait = vl, copy_state(model), 0
        else:
            wait += 1
        if verbose and (ep % 10 == 0 or ep == 1):
            print(f'    {titulo} ep {ep:3d} | val_loss {vl:.4f} val_acc {va:.4f}')
        if wait >= patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        y_prob = torch.softmax(model(tXk_te, tXc_te), dim=1).cpu().numpy()
    y_pred = y_prob.argmax(1)
    out = _metricas(dados['y_te'], y_pred)
    out.update({'_model': model, '_hist': hist, '_y_prob': y_prob, '_y_pred': y_pred})
    return out

def treinar_lightgbm(dados, optuna_trials=0):
    d = dados['lgbm']
    X_tr, y_tr   = d['X_tr'], d['y_tr']
    X_val, y_val = d['X_val'], d['y_val']
    X_test, y_test = d['X_test'], d['y_test']

    classes = np.unique(y_tr)
    pesos = compute_class_weight('balanced', classes=classes, y=y_tr)
    cwd = dict(zip(classes, pesos))
    sw = pd.Series(y_tr).map(cwd).values

    if optuna_trials > 0:
        def objective(trial):
            params = dict(
                objective='multiclass', num_class=3, verbosity=-1, n_estimators=600,
                random_state=RANDOM_STATE, n_jobs=-1,
                learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                num_leaves       = trial.suggest_int('num_leaves', 20, 128),
                max_depth        = trial.suggest_int('max_depth', 3, 12),
                min_child_samples= trial.suggest_int('min_child_samples', 10, 120),
                colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
                subsample        = trial.suggest_float('subsample', 0.5, 1.0),
                subsample_freq   = trial.suggest_int('subsample_freq', 1, 7),
                reg_alpha        = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                reg_lambda       = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            )
            m = LGBMClassifier(**params)
            m.fit(X_tr, y_tr, sample_weight=sw, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])
            return f1_score(y_val, np.argmax(m.predict_proba(X_val), axis=1), average='macro')
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=optuna_trials, show_progress_bar=False)
        params = study.best_params.copy()
        params.update(dict(objective='multiclass', num_class=3, random_state=RANDOM_STATE,
                           n_jobs=-1, verbosity=-1, n_estimators=800))
    else:
        params = dict(LGBM_PARAMS)

    model = LGBMClassifier(**params)
    model.fit(X_tr, y_tr, sample_weight=sw, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(40, verbose=False)])
    y_prob = model.predict_proba(X_test)
    y_pred = np.argmax(y_prob, axis=1)
    out = _metricas(y_test, y_pred)
    out.update({'_model': model, '_y_prob': y_prob, '_y_pred': y_pred})
    return out

def importancia_permutacao(model, dados, seed=42):
    to_t = lambda a, d: torch.tensor(a, dtype=d, device=device)
    Xc, Xk, y_te = dados['Xc_te'].copy(), dados['Xk_te'].copy(), dados['y_te']
    def mf1(xc, xk):
        model.eval()
        with torch.no_grad():
            p = model(to_t(xk, torch.long), to_t(xc, torch.float32)).argmax(1).cpu().numpy()
        return f1_score(y_te, p, average='macro', zero_division=0)
    base = mf1(Xc, Xk)
    rng = np.random.default_rng(seed)
    nomes = list(dados['cont']) + ['[cat] hora', '[cat] mes', '[cat] dia_semana']
    imp = []
    for j in range(Xc.shape[1]):
        xcp = Xc.copy(); xcp[:, j] = rng.permutation(xcp[:, j]); imp.append(base - mf1(xcp, Xk))
    for j in range(Xk.shape[1]):
        xkp = Xk.copy(); xkp[:, j] = rng.permutation(xkp[:, j]); imp.append(base - mf1(Xc, xkp))
    return pd.DataFrame({'feature': nomes, 'importancia': imp}).sort_values('importancia', ascending=False).reset_index(drop=True)

print('Funcoes de treino/avaliacao prontas | device:', device)

## 8. Loop Combinatório (24 experimentos)

Para cada **janela**, a Base Master é construída **uma vez** e reutilizada nos dois toggles de distância e nos três algoritmos (evita recomputar a FE 24 vezes — só 4). Cada execução registra uma linha em `resultados`.

> **Tempo estimado (Colab GPU):** ~15–30 min. Os transformers dominam o custo; o LightGBM é rápido. Com `LGBM_OPTUNA_TRIALS>0` o tempo do LightGBM cresce muito.

In [ ]:
resultados = []
print('Device:', device)

for window in JANELAS:
    print('=' * 60)
    print(f'JANELA: {window}')
    print('=' * 60)
    dfm = construir_base_master(window)
    print(f'  Base Master: {dfm.shape[0]:,} linhas x {dfm.shape[1]} colunas')

    for use_dist in DIST_OPTIONS:
        dados = preparar_dados(dfm, use_dist)
        tag = 'COM dist' if use_dist else 'SEM dist'
        print(f'  -> {tag} | {len(dados["cont"])} features continuas')

        def _registrar(nome, r):
            resultados.append({
                'algoritmo': nome, 'janela': window, 'dist_aeroporto': use_dist,
                'accuracy': r['accuracy'], 'macro_f1': r['macro_f1'],
                'f1_nao_evento': r['f1_nao_evento'], 'f1_clima': r['f1_clima'],
                'f1_atraso_voo': r['f1_atraso_voo'],
            })
            print(f'       {nome:15s} macro_f1={r["macro_f1"]:.4f} acc={r["accuracy"]:.4f}')

        if 'LightGBM' in ALGORITMOS:
            _registrar('LightGBM', treinar_lightgbm(dados, LGBM_OPTUNA_TRIALS))
        if 'FT-Transformer' in ALGORITMOS:
            _registrar('FT-Transformer', treinar_transformer(build_ft, 'FT-Transformer', dados))
        if 'TabTransformer' in ALGORITMOS:
            _registrar('TabTransformer', treinar_transformer(build_tab, 'TabTransformer', dados))

df_resultados = pd.DataFrame(resultados)
print('\nConcluido:', len(df_resultados), 'experimentos.')
display(df_resultados.sort_values('macro_f1', ascending=False).reset_index(drop=True))

## 9. Comparação Geral (todos contra todos)

Três visões complementares:
1. **Heatmaps** por algoritmo (janela × distância) — mostra onde cada modelo brilha.
2. **Barras agrupadas** — os 24 cenários lado a lado por algoritmo.
3. **Melhor cenário** destacado + ranking completo.

In [ ]:
# 1) Heatmaps por algoritmo (macro F1)
algs = [a for a in ALGORITMOS if a in df_resultados['algoritmo'].unique()]
fig, axes = plt.subplots(1, len(algs), figsize=(5.5*len(algs), 4))
if len(algs) == 1:
    axes = [axes]
for ax, alg in zip(axes, algs):
    piv = (df_resultados[df_resultados['algoritmo'] == alg]
           .pivot(index='janela', columns='dist_aeroporto', values='macro_f1')
           .reindex(JANELAS))
    piv = piv.rename(columns={True: 'Com Dist', False: 'Sem Dist'})
    sns.heatmap(piv, annot=True, fmt='.4f', cmap='YlGnBu', ax=ax, cbar=False)
    ax.set_title(alg); ax.set_xlabel(''); ax.set_ylabel('Janela')
plt.suptitle('Macro F1 por Janela x Distancia ao Aeroporto', y=1.03)
plt.tight_layout(); plt.show()

# 2) Barras agrupadas: todos os cenarios
dfp = df_resultados.copy()
dfp['cenario'] = dfp['janela'] + ' | ' + dfp['dist_aeroporto'].map({True: 'c/dist', False: 's/dist'})
ordem = [f'{j} | {d}' for j in JANELAS for d in ['c/dist', 's/dist']]
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=dfp, x='cenario', y='macro_f1', hue='algoritmo', order=ordem, ax=ax)
ax.set_ylim(0.70, 1.0); ax.set_title('Macro F1 - todos os cenarios')
ax.set_xlabel('Cenario (janela | distancia)'); ax.set_ylabel('Macro F1')
plt.xticks(rotation=45, ha='right'); ax.legend(title='Algoritmo'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Ranking completo + melhor cenario
ranking = df_resultados.sort_values('macro_f1', ascending=False).reset_index(drop=True)
print('=' * 60)
print('RANKING COMPLETO (por Macro F1)')
print('=' * 60)
display(ranking)

melhor = ranking.iloc[0]
print('\n' + '=' * 60)
print('MELHOR CENARIO GERAL')
print('=' * 60)
print(f"  Algoritmo : {melhor['algoritmo']}")
print(f"  Janela    : {melhor['janela']}")
print(f"  Distancia : {'COM' if melhor['dist_aeroporto'] else 'SEM'} distancia ao aeroporto")
print(f"  Macro F1  : {melhor['macro_f1']:.4f} | Accuracy: {melhor['accuracy']:.4f}")

# Melhor por algoritmo e efeito medio da distancia
print('\n--- Melhor cenario por algoritmo ---')
display(df_resultados.loc[df_resultados.groupby('algoritmo')['macro_f1'].idxmax()]
        .sort_values('macro_f1', ascending=False).reset_index(drop=True))

print('\n--- Efeito medio da distancia (macro F1 medio) ---')
display(df_resultados.groupby(['algoritmo', 'dist_aeroporto'])['macro_f1'].mean().unstack()
        .rename(columns={True: 'Com Dist', False: 'Sem Dist'}))

## 10. Explicabilidade (XAI) do Melhor Cenário

Retreina **apenas o melhor cenário geral** e roda a XAI apropriada ao algoritmo vencedor:
- **LightGBM** → SHAP (TreeExplainer), importância global por classe.
- **FT/TabTransformer** → importância por permutação (queda no Macro F1).

> Para explicar outro cenário, edite `alvo` manualmente (ex.: escolha uma linha específica de `df_resultados`).

In [ ]:
alvo = ranking.iloc[0].to_dict()   # troque por outra linha de df_resultados se quiser
print(f"XAI para: {alvo['algoritmo']} | janela {alvo['janela']} | "
      f"{'COM' if alvo['dist_aeroporto'] else 'SEM'} distancia")

dfm_alvo = construir_base_master(alvo['janela'])
dados_alvo = preparar_dados(dfm_alvo, bool(alvo['dist_aeroporto']))

if alvo['algoritmo'] == 'LightGBM':
    r = treinar_lightgbm(dados_alvo, LGBM_OPTUNA_TRIALS)
    model = r['_model']
    feats = dados_alvo['lgbm']['features']
    X_test = dados_alvo['lgbm']['X_test']
    Xs = X_test.sample(n=min(5000, len(X_test)), random_state=RANDOM_STATE).reset_index(drop=True)

    explainer = shap.TreeExplainer(model)
    raw = explainer.shap_values(Xs)
    svc = raw if isinstance(raw, list) else [raw[:, :, i] for i in range(raw.shape[2])]
    mean_abs = np.array([np.abs(s).mean(axis=0) for s in svc])
    imp = pd.DataFrame(mean_abs.T, index=feats, columns=nomes_classes)
    imp['Total'] = imp.sum(axis=1)
    imp = imp.sort_values('Total', ascending=False)
    imp.head(20)[nomes_classes].iloc[::-1].plot(kind='barh', stacked=True, figsize=(10, 8), colormap='viridis')
    plt.title(f"Importancia Global SHAP - Top 20 ({alvo['algoritmo']})")
    plt.xlabel('Impacto medio |SHAP|'); plt.tight_layout(); plt.show()
    display(imp.head(20))
else:
    build = build_ft if alvo['algoritmo'] == 'FT-Transformer' else build_tab
    r = treinar_transformer(build, alvo['algoritmo'], dados_alvo)
    imp = importancia_permutacao(r['_model'], dados_alvo)
    top = imp.head(20).iloc[::-1]
    plt.figure(figsize=(10, 8))
    plt.barh(top['feature'], top['importancia'], color='#2b8cbe')
    plt.title(f"Top 20 Importancia por Permutacao - {alvo['algoritmo']}")
    plt.xlabel('Queda no Macro-F1'); plt.tight_layout(); plt.show()
    display(imp.head(20))

# Matriz de confusao do melhor cenario
cm = confusion_matrix(dados_alvo['y_te'] if alvo['algoritmo'] != 'LightGBM' else dados_alvo['lgbm']['y_test'],
                      r['_y_pred'])
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=nomes_classes, yticklabels=nomes_classes)
plt.title(f"Matriz de Confusao - {alvo['algoritmo']} (melhor cenario)")
plt.ylabel('Classe Real'); plt.xlabel('Classe Prevista'); plt.tight_layout(); plt.show()

## 11. Conclusões

Como ler os resultados deste notebook:

- **Melhor algoritmo:** compare a coluna `macro_f1` no ranking (célula 9). O Macro F1 pondera as três classes igualmente — apropriado para dados balanceados 1:1:1.
- **Efeito da janela:** dentro de cada algoritmo, veja se janelas maiores (4h) ou menores (1h) ajudam. Janelas maiores agregam mais contexto climático, mas diluem a granularidade temporal.
- **Efeito da distância ao aeroporto:** a tabela "Efeito médio da distância" quantifica o ganho de incluir o sinal espacial. Se `Com Dist` > `Sem Dist` de forma consistente, confirma a hipótese de que a **proximidade do aeroporto é discriminativa** — especialmente para separar Clima vs Atraso de Voo.
- **Confusão Clima × Atraso de Voo:** observada nos experimentos anteriores; a matriz de confusão do melhor cenário (célula 10) mostra se o vencedor mitiga essa fronteira ou apenas a desloca.

**Ressalvas metodológicas (para a tese):**
- Todos os modelos usam o **mesmo** split temporal, features e pesos de classe balanceados — a comparação isola o efeito do algoritmo.
- Os transformers usam capacidade fixa (dim=32, depth=6, heads=8); o LightGBM usa params fixos (ou Optuna se ativado). Nenhum passou por busca exaustiva de arquitetura — os resultados refletem configurações razoáveis, não ótimos globais.
- Execução em Google Colab; a ingestão de dados (GCS/DuckDB/Drive) precisa de adaptação para rodar localmente.